In [ ]:
# =============================================================================
# CELL 1: IMPORTS AND BASIC SETTINGS
# Assignment: Reproduce MAGNET results on BRCA, BLCA, OV
# =============================================================================

import os
import sys
import glob
import time
import shutil
import subprocess
import pandas as pd
import numpy as np

from getpass import getpass
from IPython.display import display

print("=" * 80)
print("CELL 1: IMPORTS AND BASIC SETTINGS")
print("=" * 80)

# -----------------------------------------------------------------------------
# GitHub repositories
# -----------------------------------------------------------------------------
OFFICIAL_MAGNET_REPO = "https://github.com/SinaTabakhi/MAGNET.git"
MY_REPO_URL = "https://github.com/goutamsaums/mlsi-summer-internship-2026.git"

# -----------------------------------------------------------------------------
# Local Colab paths
# -----------------------------------------------------------------------------
MAGNET_DIR = "/content/MAGNET"
MY_REPO_DIR = "/content/mlsi-summer-internship-2026"

MY_SPLIT_DATA_DIR = f"{MY_REPO_DIR}/MAGNET_Datasets/split_data"
MAGNET_SPLIT_DATA_DIR = f"{MAGNET_DIR}/data/split_data"

# -----------------------------------------------------------------------------
# Paper accuracy values from MAGNET paper Table 1
# -----------------------------------------------------------------------------
paper_accuracy = {
    "BRCA": 0.918,
    "BLCA": 0.970,
    "OV": 0.614
}

# -----------------------------------------------------------------------------
# Dataset-specific config files
# -----------------------------------------------------------------------------
configs = {
    "BRCA": "configs/MAGNET_BRCA.yaml",
    "BLCA": "configs/MAGNET_BLCA.yaml",
    "OV": "configs/MAGNET_OV.yaml"
}

RESULT_FOLDER = "MAGNET_Reproduction_BRCA_BLCA_OV"
RESULT_DIR = os.path.join(MY_REPO_DIR, RESULT_FOLDER)

print("Official MAGNET repo:", OFFICIAL_MAGNET_REPO)
print("My GitHub repo      :", MY_REPO_URL)
print("MAGNET directory    :", MAGNET_DIR)
print("My repo directory   :", MY_REPO_DIR)
print("Result folder       :", RESULT_FOLDER)

print("\nPaper accuracy values:")
for k, v in paper_accuracy.items():
    print(f"{k}: {v}")

print("\nCELL 1 COMPLETE")

CELL 1: IMPORTS AND BASIC SETTINGS
Official MAGNET repo: https://github.com/SinaTabakhi/MAGNET.git
My GitHub repo      : https://github.com/goutamsaums/mlsi-summer-internship-2026.git
MAGNET directory    : /content/MAGNET
My repo directory   : /content/mlsi-summer-internship-2026
Result folder       : MAGNET_Reproduction_BRCA_BLCA_OV

Paper accuracy values:
BRCA: 0.918
BLCA: 0.97
OV: 0.614

CELL 1 COMPLETE


In [ ]:
# =============================================================================
# CELL 2: CHECK HARDWARE, CLEAN FOLDERS, CLONE REPOSITORIES
# =============================================================================

print("=" * 80)
print("CELL 2: HARDWARE CHECK + CLONE REPOSITORIES")
print("=" * 80)

# -----------------------------------------------------------------------------
# Check GPU / CPU
# -----------------------------------------------------------------------------
print("\nChecking runtime hardware...")

try:
    import torch
    print("PyTorch version :", torch.__version__)
    print("CUDA available  :", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU             :", torch.cuda.get_device_name(0))
        print("CUDA version    :", torch.version.cuda)
    else:
        print("Running on CPU. It will work, but may be slower.")

except Exception as e:
    print("Torch check failed:", e)

# -----------------------------------------------------------------------------
# Clean old folders
# -----------------------------------------------------------------------------
print("\nCleaning old Colab folders...")

os.chdir("/content")

for folder in [MAGNET_DIR, MY_REPO_DIR]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print("Removed:", folder)

# -----------------------------------------------------------------------------
# Clone official MAGNET repository
# -----------------------------------------------------------------------------
print("\nCloning official MAGNET repository...")

subprocess.run(
    ["git", "clone", OFFICIAL_MAGNET_REPO, MAGNET_DIR],
    check=True
)

print("Official MAGNET cloned at:", MAGNET_DIR)

# -----------------------------------------------------------------------------
# Clone my GitHub repository
# -----------------------------------------------------------------------------
print("\nCloning my GitHub repository...")

subprocess.run(
    ["git", "clone", MY_REPO_URL, MY_REPO_DIR],
    check=True
)

print("My repository cloned at:", MY_REPO_DIR)

print("\nCELL 2 COMPLETE")

CELL 2: HARDWARE CHECK + CLONE REPOSITORIES

Checking runtime hardware...
PyTorch version : 2.11.0+cpu
CUDA available  : False
Running on CPU. It will work, but may be slower.

Cleaning old Colab folders...
Removed: /content/MAGNET
Removed: /content/mlsi-summer-internship-2026

Cloning official MAGNET repository...
Official MAGNET cloned at: /content/MAGNET

Cloning my GitHub repository...
My repository cloned at: /content/mlsi-summer-internship-2026

CELL 2 COMPLETE


In [ ]:
# =============================================================================
# CELL 3: COPY DATASETS FROM MY GITHUB + INSTALL DEPENDENCIES
# =============================================================================

print("=" * 80)
print("CELL 3: DATASET PREPARATION + DEPENDENCY INSTALLATION")
print("=" * 80)

# -----------------------------------------------------------------------------
# Copy copied MAGNET datasets from my repo into official MAGNET folder
# -----------------------------------------------------------------------------
print("\nPreparing MAGNET datasets...")

if os.path.exists(MY_SPLIT_DATA_DIR):
    print("Found dataset in my GitHub repo:")
    print(MY_SPLIT_DATA_DIR)

    if os.path.exists(MAGNET_SPLIT_DATA_DIR):
        shutil.rmtree(MAGNET_SPLIT_DATA_DIR)

    shutil.copytree(MY_SPLIT_DATA_DIR, MAGNET_SPLIT_DATA_DIR)
    print("Copied my GitHub split_data into official MAGNET folder.")

else:
    print("WARNING: Dataset folder not found in my GitHub repo.")
    print("Using official MAGNET repo data/split_data instead.")

# -----------------------------------------------------------------------------
# Install Python dependencies
# -----------------------------------------------------------------------------
print("\nInstalling required Python packages...")

os.chdir(MAGNET_DIR)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "lightning==2.1.3",
        "yacs",
        "umap-learn",
        "comet_ml",
        "scikit-learn",
        "pandas",
        "numpy",
        "matplotlib",
        "torch-geometric"
    ],
    check=True
)

# -----------------------------------------------------------------------------
# Optional PyTorch Geometric compiled packages
# -----------------------------------------------------------------------------
print("\nInstalling/checking PyTorch Geometric compiled packages...")

try:
    import torch

    TORCH_VERSION = torch.__version__.split("+")[0]

    if torch.cuda.is_available() and torch.version.cuda is not None:
        CUDA_VERSION = "cu" + torch.version.cuda.replace(".", "")
    else:
        CUDA_VERSION = "cpu"

    pyg_wheel_url = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html"

    print("Torch version for PyG wheels:", TORCH_VERSION)
    print("CUDA version for PyG wheels :", CUDA_VERSION)
    print("PyG wheel URL               :", pyg_wheel_url)

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pyg_lib",
            "torch_scatter",
            "torch_sparse",
            "torch_cluster",
            "torch_spline_conv",
            "-f", pyg_wheel_url
        ],
        check=False
    )

except Exception as e:
    print("PyG compiled package installation skipped/failed:")
    print(e)

print("\nCELL 3 COMPLETE")

CELL 3: DATASET PREPARATION + DEPENDENCY INSTALLATION

Preparing MAGNET datasets...
Found dataset in my GitHub repo:
/content/mlsi-summer-internship-2026/MAGNET_Datasets/split_data
Copied my GitHub split_data into official MAGNET folder.

Installing required Python packages...

Installing/checking PyTorch Geometric compiled packages...
Torch version for PyG wheels: 2.11.0
CUDA version for PyG wheels : cpu
PyG wheel URL               : https://data.pyg.org/whl/torch-2.11.0+cpu.html

CELL 3 COMPLETE


In [ ]:
# =============================================================================
# CELL 4: VERIFY INSTALLATION, CONFIGS, DATASETS AND FILES
# =============================================================================

print("=" * 80)
print("CELL 4: VERIFICATION")
print("=" * 80)

# -----------------------------------------------------------------------------
# Verify imports
# -----------------------------------------------------------------------------
print("\nVerifying important imports...")

try:
    import torch
    import torch_geometric
    import lightning
    import yacs

    print("torch           :", torch.__version__)
    print("torch_geometric :", torch_geometric.__version__)
    print("lightning       :", lightning.__version__)
    print("yacs            : imported successfully")
    print("Import verification complete.")

except Exception as e:
    print("Import verification failed.")
    print(e)
    raise

# -----------------------------------------------------------------------------
# Verify folder structure
# -----------------------------------------------------------------------------
print("\nVerifying MAGNET folder structure...")

os.chdir(MAGNET_DIR)

print("\nCurrent folder:")
print(os.getcwd())

print("\nMain files:")
for f in ["main_inference.py", "main_data_split.py", "main_tune.py"]:
    print(f, ":", "FOUND" if os.path.exists(f) else "MISSING")

print("\nConfig files:")
for dataset, cfg in configs.items():
    print(dataset, cfg, ":", "FOUND" if os.path.exists(cfg) else "MISSING")

print("\nDataset folders:")
for dataset in ["BRCA", "BLCA", "OV"]:
    dataset_path = f"{MAGNET_SPLIT_DATA_DIR}/{dataset}"
    print(dataset, ":", "FOUND" if os.path.exists(dataset_path) else "MISSING")

    if os.path.exists(dataset_path):
        seed_folders = sorted([
            x for x in os.listdir(dataset_path)
            if os.path.isdir(os.path.join(dataset_path, x))
        ])
        print("   split folders:", seed_folders)

# -----------------------------------------------------------------------------
# Check required files in split 10
# -----------------------------------------------------------------------------
print("\nChecking required files in each dataset split 10...")

required_files = [
    "DNA.csv",
    "mRNA.csv",
    "miRNA.csv",
    "label_train_paired.csv",
    "label_val_paired.csv",
    "label_test_paired.csv",
    "label_train_unpaired.csv",
    "label_val_unpaired.csv",
    "label_test_unpaired.csv"
]

check_rows = []

for dataset in ["BRCA", "BLCA", "OV"]:
    split10_path = f"{MAGNET_SPLIT_DATA_DIR}/{dataset}/10"

    for file in required_files:
        file_path = f"{split10_path}/{file}"
        check_rows.append({
            "Dataset": dataset,
            "Split": "10",
            "File": file,
            "Exists": os.path.exists(file_path)
        })

check_df = pd.DataFrame(check_rows)
display(check_df)

if not check_df["Exists"].all():
    missing = check_df[check_df["Exists"] == False]
    print("\nMissing files detected:")
    display(missing)
    raise FileNotFoundError("Some required MAGNET dataset files are missing.")

print("\nCELL 4 COMPLETE")
print("MAGNET setup and datasets are ready.")

CELL 4: VERIFICATION

Verifying important imports...
torch           : 2.11.0+cpu
torch_geometric : 2.8.0
lightning       : 2.1.3
yacs            : imported successfully
Import verification complete.

Verifying MAGNET folder structure...

Current folder:
/content/MAGNET

Main files:
main_inference.py : FOUND
main_data_split.py : FOUND
main_tune.py : FOUND

Config files:
BRCA configs/MAGNET_BRCA.yaml : FOUND
BLCA configs/MAGNET_BLCA.yaml : FOUND
OV configs/MAGNET_OV.yaml : FOUND

Dataset folders:
BRCA : FOUND
   split folders: ['10', '20', '30', '40', '50']
BLCA : FOUND
   split folders: ['10', '20', '30', '40', '50']
OV : FOUND
   split folders: ['10', '20', '30', '40', '50']

Checking required files in each dataset split 10...


,Dataset,Split,File,Exists
0,BRCA,10,DNA.csv,True
1,BRCA,10,mRNA.csv,True
2,BRCA,10,miRNA.csv,True
3,BRCA,10,label_train_paired.csv,True
4,BRCA,10,label_val_paired.csv,True
5,BRCA,10,label_test_paired.csv,True
6,BRCA,10,label_train_unpaired.csv,True
7,BRCA,10,label_val_unpaired.csv,True
8,BRCA,10,label_test_unpaired.csv,True
9,BLCA,10,DNA.csv,True



CELL 4 COMPLETE
MAGNET setup and datasets are ready.


In [ ]:
# =============================================================================
# CELL 5: RUN MAGNET ON BRCA, BLCA AND OV
# =============================================================================

print("=" * 80)
print("CELL 5: RUN MAGNET REPRODUCTION")
print("=" * 80)

if not os.path.exists(MAGNET_DIR):
    raise FileNotFoundError("MAGNET folder not found. Please run previous cells first.")

os.chdir(MAGNET_DIR)

if not os.path.exists("main_inference.py"):
    raise FileNotFoundError("main_inference.py not found in MAGNET folder.")

print("Current directory:", os.getcwd())
print("MAGNET setup found successfully.")

run_logs = {}
run_status = {}
total_start = time.time()

for dataset, cfg in configs.items():

    print("\n" + "=" * 80)
    print(f"RUNNING MAGNET FOR {dataset}")
    print("=" * 80)

    if not os.path.exists(cfg):
        print(f"Config file missing: {cfg}")
        run_status[dataset] = "Config missing"
        run_logs[dataset] = f"Config file missing: {cfg}"
        continue

    dataset_start = time.time()

    cmd = [
        sys.executable,
        "-u",
        "main_inference.py",
        "--cfg",
        cfg
    ]

    print("Command:", " ".join(cmd))

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    log_lines = []

    for line in process.stdout:
        print(line, end="")
        log_lines.append(line)

    process.wait()

    elapsed = time.time() - dataset_start

    run_logs[dataset] = "".join(log_lines)
    run_status[dataset] = "Success" if process.returncode == 0 else f"Failed: {process.returncode}"

    print("\nFinished:", dataset)
    print("Return code:", process.returncode)
    print("Time taken:", round(elapsed, 2), "seconds")

print("\n" + "=" * 80)
print("ALL RUNS FINISHED")
print("Total time:", round(time.time() - total_start, 2), "seconds")
print("=" * 80)

print("\nRun status:")
for dataset, status in run_status.items():
    print(dataset, ":", status)

print("\nCELL 5 COMPLETE")

Streaming output truncated to the last 5000 lines.
Validation DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 2020.38it/s]

                                                                        
Epoch 96: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s, v_num=0, train_kl_loss=0.0736, train_cls_loss=0.00927, train_total_loss=0.0166, train_acc=0.997, train_auroc=1.000, train_auprc=1.000, train_f1=0.996, train_mcc=0.993]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

Validation DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 1738.21it/s]

                                                                        
Epoch 97: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s, v_num=0, train_kl_loss=0.0803, train_cls_loss=0.00811, train_total_loss=0.0161, train_acc=1.000, train_auroc=1.000, train_auprc=1.000, train_f1=1.000, train_mcc=1.000]

Validation: |          | 0/? [00:00<?, ?it/

In [11]:
# =============================================================================
# CELL 6: EXTRACT ACCURACY FROM MAGNET OUTPUT FILES
# =============================================================================

print("=" * 80)
print("CELL 6: EXTRACT ACCURACY RESULTS")
print("=" * 80)

results = []
metric_summaries = {}

for dataset in ["BRCA", "BLCA", "OV"]:

    print("\n" + "-" * 80)
    print(f"Extracting results for {dataset}")
    print("-" * 80)

    output_dir = os.path.join(MAGNET_DIR, "output", dataset)

    candidate_files = []

    if os.path.exists(output_dir):
        candidate_files.extend(
            glob.glob(os.path.join(output_dir, "**", "*.csv"), recursive=True)
        )

    if len(candidate_files) == 0:
        candidate_files.extend(
            glob.glob(os.path.join(MAGNET_DIR, "output", "**", "*.csv"), recursive=True)
        )

    print("CSV files found:", len(candidate_files))

    valid_files = []

    for file_path in candidate_files:
        try:
            temp = pd.read_csv(file_path, nrows=5)
            if {"metric", "value"}.issubset(temp.columns):
                valid_files.append(file_path)
        except Exception:
            pass

    dataset_valid_files = [
        f for f in valid_files
        if dataset.lower() in f.lower()
    ]

    if len(dataset_valid_files) > 0:
        valid_files = dataset_valid_files

    if len(valid_files) == 0:
        print("No valid metric CSV found for", dataset)

        results.append({
            "Dataset": dataset,
            "My Accuracy": np.nan,
            "My Std": np.nan,
            "Paper Accuracy": paper_accuracy[dataset],
            "Difference": np.nan,
            "Tolerance": 0.06 if dataset == "OV" else 0.03,
            "Roughly Matches?": "No result",
            "Run Status": run_status.get(dataset, "Unknown"),
            "Result File": "Not found"
        })

        continue

    sorted_files = [
        f for f in valid_files
        if f.endswith("_cls_sorted.csv")
    ]

    if sorted_files:
        result_file = sorted_files[0]
    else:
        result_file = sorted(valid_files, key=os.path.getmtime, reverse=True)[0]

    print("Selected result file:", result_file)

    df = pd.read_csv(result_file)

    summary = (
        df.groupby("metric")["value"]
        .agg(["mean", "std", "min", "max"])
        .round(4)
    )

    metric_summaries[dataset] = summary

    print("\nMetric summary:")
    display(summary)

    if "acc" in summary.index:
        my_acc = float(summary.loc["acc", "mean"])
        my_std = float(summary.loc["acc", "std"])
    else:
        my_acc = np.nan
        my_std = np.nan

    paper_acc = paper_accuracy[dataset]
    tolerance = 0.06 if dataset == "OV" else 0.03

    if pd.isna(my_acc):
        diff = np.nan
        roughly_match = "No"
    else:
        diff = my_acc - paper_acc
        roughly_match = "Yes" if abs(diff) <= tolerance else "No"

    results.append({
        "Dataset": dataset,
        "My Accuracy": my_acc,
        "My Std": my_std,
        "Paper Accuracy": paper_acc,
        "Difference": diff,
        "Tolerance": tolerance,
        "Roughly Matches?": roughly_match,
        "Run Status": run_status.get(dataset, "Unknown"),
        "Result File": result_file
    })

print("\nCELL 6 COMPLETE")

CELL 6: EXTRACT ACCURACY RESULTS

--------------------------------------------------------------------------------
Extracting results for BRCA
--------------------------------------------------------------------------------
CSV files found: 13
Selected result file: /content/MAGNET/output/BRCA/MAGNET_BRCA_cls_sorted.csv

Metric summary:


,mean,std,min,max
metric,,,,
acc,0.9198,0.0131,0.9010,0.9375
cls_loss,0.2774,0.0506,0.2068,0.3247
f1_macro,0.9047,0.0174,0.8823,0.9298
f1_micro,0.9198,0.0131,0.9010,0.9375
f1_weighted,0.9193,0.0129,0.9013,0.9367
kl_loss,0.3337,0.0255,0.2940,0.3557
mcc,0.8869,0.0189,0.8589,0.9114
paired_acc,0.9200,0.0183,0.9000,0.9333
paired_f1_macro,0.8925,0.0289,0.8474,0.9224



--------------------------------------------------------------------------------
Extracting results for BLCA
--------------------------------------------------------------------------------
CSV files found: 13
Selected result file: /content/MAGNET/output/BLCA/MAGNET_BLCA_cls_sorted.csv

Metric summary:


,mean,std,min,max
metric,,,,
acc,0.9705,0.0062,0.9659,0.9773
auprc,0.7328,0.1180,0.5506,0.8542
auroc,0.9565,0.0302,0.9256,0.9911
cls_loss,0.1368,0.0585,0.0602,0.2242
f1,0.6262,0.0806,0.5714,0.7500
kl_loss,0.1513,0.0236,0.1109,0.1718
mcc,0.6236,0.0877,0.5603,0.7381
paired_acc,0.9690,0.0065,0.9643,0.9762
paired_auprc,0.7397,0.1270,0.5506,0.8875



--------------------------------------------------------------------------------
Extracting results for OV
--------------------------------------------------------------------------------
CSV files found: 13
Selected result file: /content/MAGNET/output/OV/MAGNET_OV_cls_sorted.csv

Metric summary:


,mean,std,min,max
metric,,,,
acc,0.5753,0.0750,0.4795,0.6575
auprc,0.6229,0.0431,0.5476,0.6528
auroc,0.6332,0.0598,0.5668,0.7117
cls_loss,2.0376,0.3567,1.6649,2.4982
f1,0.5772,0.0545,0.5250,0.6479
kl_loss,0.0872,0.0126,0.0775,0.1085
mcc,0.1512,0.1495,-0.0391,0.3148
paired_acc,0.5944,0.0697,0.5000,0.6944
paired_auprc,0.6326,0.0916,0.4881,0.7170



CELL 6 COMPLETE


In [12]:
# =============================================================================
# CELL 7: CREATE FINAL DELIVERABLE TABLE
# =============================================================================

print("=" * 80)
print("CELL 7: FINAL DELIVERABLE TABLE")
print("=" * 80)

final_table = pd.DataFrame(results)

clean_table = final_table[
    ["Dataset", "My Accuracy", "Paper Accuracy", "Roughly Matches?"]
].copy()

clean_table["My Accuracy"] = clean_table["My Accuracy"].round(4)
clean_table["Paper Accuracy"] = clean_table["Paper Accuracy"].round(3)

print("\nFINAL DELIVERABLE TABLE:")
display(clean_table)

print("\nDetailed table:")
display(final_table)

print("\nMarkdown table:")
print("| Dataset | My Accuracy | Paper Accuracy | Roughly Matches? |")
print("|---|---:|---:|---|")

for _, row in clean_table.iterrows():
    acc_text = "NA" if pd.isna(row["My Accuracy"]) else f"{row['My Accuracy']:.4f}"
    print(
        f"| {row['Dataset']} | {acc_text} | "
        f"{row['Paper Accuracy']:.3f} | {row['Roughly Matches?']} |"
    )

print("\nCELL 7 COMPLETE")

CELL 7: FINAL DELIVERABLE TABLE

FINAL DELIVERABLE TABLE:


,Dataset,My Accuracy,Paper Accuracy,Roughly Matches?
0,BRCA,0.9198,0.918,Yes
1,BLCA,0.9705,0.970,Yes
2,OV,0.5753,0.614,Yes



Detailed table:


,Dataset,My Accuracy,My Std,Paper Accuracy,Difference,Tolerance,Roughly Matches?,Run Status,Result File
0,BRCA,0.9198,0.0131,0.918,0.0018,0.03,Yes,Success,/content/MAGNET/output/BRCA/MAGNET_BRCA_cls_so...
1,BLCA,0.9705,0.0062,0.970,0.0005,0.03,Yes,Success,/content/MAGNET/output/BLCA/MAGNET_BLCA_cls_so...
2,OV,0.5753,0.0750,0.614,-0.0387,0.06,Yes,Success,/content/MAGNET/output/OV/MAGNET_OV_cls_sorted...



Markdown table:
| Dataset | My Accuracy | Paper Accuracy | Roughly Matches? |
|---|---:|---:|---|
| BRCA | 0.9198 | 0.918 | Yes |
| BLCA | 0.9705 | 0.970 | Yes |
| OV | 0.5753 | 0.614 | Yes |

CELL 7 COMPLETE


In [13]:
# =============================================================================
# CELL 8: SAVE RESULT FILES, LOGS, README AND NOTES
# =============================================================================

print("=" * 80)
print("CELL 8: SAVE RESULTS")
print("=" * 80)

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "metric_summaries"), exist_ok=True)

# -----------------------------------------------------------------------------
# Save tables
# -----------------------------------------------------------------------------
clean_csv = os.path.join(RESULT_DIR, "MAGNET_reproduction_table_clean.csv")
detailed_csv = os.path.join(RESULT_DIR, "MAGNET_reproduction_table_detailed.csv")

clean_table.to_csv(clean_csv, index=False)
final_table.to_csv(detailed_csv, index=False)

print("Saved:", clean_csv)
print("Saved:", detailed_csv)

# -----------------------------------------------------------------------------
# Save run logs
# -----------------------------------------------------------------------------
for dataset, log_text in run_logs.items():
    log_path = os.path.join(RESULT_DIR, "logs", f"{dataset}_run_log.txt")
    with open(log_path, "w", encoding="utf-8") as f:
        f.write(log_text)
    print("Saved log:", log_path)

# -----------------------------------------------------------------------------
# Save metric summaries
# -----------------------------------------------------------------------------
for dataset, summary in metric_summaries.items():
    summary_path = os.path.join(RESULT_DIR, "metric_summaries", f"{dataset}_metric_summary.csv")
    summary.to_csv(summary_path)
    print("Saved metric summary:", summary_path)

# -----------------------------------------------------------------------------
# Create README.md
# -----------------------------------------------------------------------------
readme_path = os.path.join(RESULT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write("# MAGNET Reproduction: BRCA, BLCA and OV\n\n")
    f.write("This folder contains reproduction results for the official MAGNET model on BRCA, BLCA and OV datasets.\n\n")

    f.write("## Aim\n\n")
    f.write("To run the official MAGNET code on all three datasets and check whether the obtained accuracy comes close to the values reported in the MAGNET paper.\n\n")

    f.write("## Dataset Source\n\n")
    f.write("The split datasets were copied from the official MAGNET repository and saved in this repository under:\n\n")
    f.write("`MAGNET_Datasets/split_data/`\n\n")

    f.write("## Paper Accuracy Values\n\n")
    f.write("| Dataset | Paper Accuracy |\n")
    f.write("|---|---:|\n")
    f.write("| BRCA | 0.918 |\n")
    f.write("| BLCA | 0.970 |\n")
    f.write("| OV | 0.614 |\n\n")

    f.write("## Final Reproduction Table\n\n")
    f.write("| Dataset | My Accuracy | Paper Accuracy | Roughly Matches? |\n")
    f.write("|---|---:|---:|---|\n")

    for _, row in clean_table.iterrows():
        acc_text = "NA" if pd.isna(row["My Accuracy"]) else f"{row['My Accuracy']:.4f}"
        f.write(
            f"| {row['Dataset']} | {acc_text} | "
            f"{row['Paper Accuracy']:.3f} | {row['Roughly Matches?']} |\n"
        )

    f.write("\n## Files\n\n")
    f.write("- `MAGNET_reproduction_table_clean.csv`: final 3-row deliverable table\n")
    f.write("- `MAGNET_reproduction_table_detailed.csv`: detailed table with standard deviation, difference and tolerance\n")
    f.write("- `logs/`: run logs for BRCA, BLCA and OV\n")
    f.write("- `metric_summaries/`: metric summaries extracted from MAGNET output files\n")

print("Saved README:", readme_path)

# -----------------------------------------------------------------------------
# Create notes.md
# -----------------------------------------------------------------------------
notes_path = os.path.join(RESULT_DIR, "notes.md")

with open(notes_path, "w", encoding="utf-8") as f:
    f.write("# Notes on MAGNET Reproduction\n\n")

    f.write("## What was done\n\n")
    f.write("I cloned the official MAGNET code and ran the official inference script on BRCA, BLCA and OV datasets.\n\n")

    f.write("## Why this was done\n\n")
    f.write("Before modifying or improving MAGNET, it is necessary to reproduce the original paper results and check that the code and dataset setup are working correctly.\n\n")

    f.write("## Commands used\n\n")
    f.write("```bash\n")
    f.write("python main_inference.py --cfg configs/MAGNET_BRCA.yaml\n")
    f.write("python main_inference.py --cfg configs/MAGNET_BLCA.yaml\n")
    f.write("python main_inference.py --cfg configs/MAGNET_OV.yaml\n")
    f.write("```\n\n")

    f.write("## Final deliverable\n\n")
    f.write("The final deliverable is the 3-row reproduction table comparing my obtained accuracy with the paper accuracy.\n\n")

    f.write("## Next possible work\n\n")
    f.write("- weighted modality fusion\n")
    f.write("- modality contribution analysis\n")
    f.write("- masking versus imputation\n")
    f.write("- improved graph edge construction\n")

print("Saved notes:", notes_path)

print("\nFiles saved in:")
print(RESULT_DIR)

print("\nCELL 8 COMPLETE")

CELL 8: SAVE RESULTS
Saved: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/MAGNET_reproduction_table_clean.csv
Saved: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/MAGNET_reproduction_table_detailed.csv
Saved log: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/logs/BRCA_run_log.txt
Saved log: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/logs/BLCA_run_log.txt
Saved log: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/logs/OV_run_log.txt
Saved metric summary: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/metric_summaries/BRCA_metric_summary.csv
Saved metric summary: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/metric_summaries/BLCA_metric_summary.csv
Saved metric summary: /content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/metric_summaries/OV_metric_summary.csv
Saved README: /content/mlsi-summer-internship-

In [14]:
# =============================================================================
# CELL 9: AUTO-PUSH RESULT FOLDER TO GITHUB
# =============================================================================

print("=" * 80)
print("CELL 9: AUTO-PUSH RESULTS TO GITHUB")
print("=" * 80)

github_username = "goutamsaums"
github_token = getpass("Enter GitHub Personal Access Token: ")

repo_url_with_token = MY_REPO_URL.replace(
    "https://",
    f"https://{github_username}:{github_token}@"
)

os.chdir(MY_REPO_DIR)

try:
    # Set remote URL with token
    subprocess.run(
        ["git", "remote", "set-url", "origin", repo_url_with_token],
        check=True
    )

    # Git identity
    subprocess.run(["git", "config", "user.name", "GOUTAM ANAND"], check=True)
    subprocess.run(["git", "config", "user.email", "goutam@students.sau.ac.in"], check=True)

    # Add result folder
    subprocess.run(["git", "add", RESULT_FOLDER], check=True)

    # Add dataset folder also, in case any local copy change remains
    subprocess.run(["git", "add", "MAGNET_Datasets"], check=False)

    # Check status
    status = subprocess.run(
        ["git", "status", "--porcelain"],
        capture_output=True,
        text=True
    )

    if status.stdout.strip():
        commit_msg = "Add MAGNET reproduction results for BRCA BLCA OV"
        subprocess.run(["git", "commit", "-m", commit_msg], check=True)
        subprocess.run(["git", "push"], check=True)

        print("Results pushed to GitHub successfully.")
        print("GitHub repo:", MY_REPO_URL)

    else:
        print("No changes to commit.")

finally:
    # Reset remote URL without token
    subprocess.run(
        ["git", "remote", "set-url", "origin", MY_REPO_URL],
        check=False
    )

print("\nCELL 9 COMPLETE")

CELL 9: AUTO-PUSH RESULTS TO GITHUB
Enter GitHub Personal Access Token: ··········
Results pushed to GitHub successfully.
GitHub repo: https://github.com/goutamsaums/mlsi-summer-internship-2026.git

CELL 9 COMPLETE


In [17]:
# =============================================================================
# CELL 10: FINAL CONFIRMATION
# =============================================================================

print("=" * 80)
print("CELL 10: FINAL CONFIRMATION")
print("=" * 80)

print("Final result folder:")
print(RESULT_DIR)

print("\nGitHub folder:")
print(f"https://github.com/goutamsaums/mlsi-summer-internship-2026/tree/main/{RESULT_FOLDER}")

print("\nFinal deliverable table:")
display(clean_table)

print("\nMarkdown table:")
print("| Dataset | My Accuracy | Paper Accuracy | Roughly Matches? |")
print("|---|---:|---:|---|")

for _, row in clean_table.iterrows():
    acc_text = "NA" if pd.isna(row["My Accuracy"]) else f"{row['My Accuracy']:.4f}"
    print(
        f"| {row['Dataset']} | {acc_text} | "
        f"{row['Paper Accuracy']:.3f} | {row['Roughly Matches?']} |"
    )

print("\ncomplete.")
print("=" * 80)

CELL 10: FINAL CONFIRMATION
Final result folder:
/content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV

GitHub folder:
https://github.com/goutamsaums/mlsi-summer-internship-2026/tree/main/MAGNET_Reproduction_BRCA_BLCA_OV

Final deliverable table:


,Dataset,My Accuracy,Paper Accuracy,Roughly Matches?
0,BRCA,0.9198,0.918,Yes
1,BLCA,0.9705,0.970,Yes
2,OV,0.5753,0.614,Yes



Markdown table:
| Dataset | My Accuracy | Paper Accuracy | Roughly Matches? |
|---|---:|---:|---|
| BRCA | 0.9198 | 0.918 | Yes |
| BLCA | 0.9705 | 0.970 | Yes |
| OV | 0.5753 | 0.614 | Yes |

complete.


In [20]:
# =============================================================================
# CELL 11: GENERATE PDF COMPARISON REPORT AND SAVE TO GITHUB
# Deliverable: PDF with heading + final MAGNET reproduction accuracy table
# =============================================================================

import os
import subprocess
import pandas as pd
from getpass import getpass

print("=" * 80)
print("CELL 11: GENERATE PDF COMPARISON REPORT AND PUSH TO GITHUB")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. Use existing paths from previous cells, or define again if needed
# -----------------------------------------------------------------------------
try:
    MY_REPO_DIR
except NameError:
    MY_REPO_DIR = "/content/mlsi-summer-internship-2026"

try:
    MY_REPO_URL
except NameError:
    MY_REPO_URL = "https://github.com/goutamsaums/mlsi-summer-internship-2026.git"

RESULT_FOLDER = "MAGNET_Reproduction_BRCA_BLCA_OV"
RESULT_DIR = os.path.join(MY_REPO_DIR, RESULT_FOLDER)

os.makedirs(RESULT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Use final table from previous cells, or recreate it if needed
# -----------------------------------------------------------------------------
if "clean_table" not in globals():
    clean_table = pd.DataFrame({
        "Dataset": ["BRCA", "BLCA", "OV"],
        "My Accuracy": [0.9198, 0.9705, 0.5753],
        "Paper Accuracy": [0.918, 0.970, 0.614],
        "Roughly Matches?": ["Yes", "Yes", "Yes"]
    })

print("\nFinal table to be written in PDF:")
display(clean_table)

# -----------------------------------------------------------------------------
# 3. Install ReportLab for PDF generation
# -----------------------------------------------------------------------------
print("\nInstalling ReportLab if needed...")

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "reportlab"],
    check=True
)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer

# -----------------------------------------------------------------------------
# 4. Create PDF file
# -----------------------------------------------------------------------------
pdf_path = os.path.join(RESULT_DIR, "MAGNET_Reproduction_Accuracy_Report.pdf")

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50,
    leftMargin=50,
    topMargin=60,
    bottomMargin=50
)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="TitleStyle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=16
)

subtitle_style = ParagraphStyle(
    name="SubtitleStyle",
    parent=styles["Normal"],
    alignment=TA_CENTER,
    fontSize=10,
    leading=14,
    spaceAfter=20
)

note_heading_style = ParagraphStyle(
    name="NoteHeadingStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10,
    leading=13,
    spaceBefore=18,
    spaceAfter=6,
    fontName="Helvetica-Bold"
)

note_style = ParagraphStyle(
    name="NoteStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=9,
    leading=13,
    spaceAfter=6
)

story = []

# Main heading
story.append(Paragraph("Comparison Report", title_style))


story.append(Spacer(1, 0.2 * inch))

# -----------------------------------------------------------------------------
# 5. Table data
# -----------------------------------------------------------------------------
table_data = [
    ["Dataset", "My Accuracy", "Paper Accuracy", "Roughly Matches?"]
]

for _, row in clean_table.iterrows():
    table_data.append([
        str(row["Dataset"]),
        f"{float(row['My Accuracy']):.4f}",
        f"{float(row['Paper Accuracy']):.3f}",
        str(row["Roughly Matches?"])
    ])

table = Table(
    table_data,
    colWidths=[1.3 * inch, 1.5 * inch, 1.6 * inch, 1.8 * inch],
    hAlign="CENTER"
)

table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
    ("FONTSIZE", (0, 0), (-1, -1), 10),
    ("BOTTOMPADDING", (0, 0), (-1, 0), 10),
    ("TOPPADDING", (0, 0), (-1, 0), 10),
    ("BOTTOMPADDING", (0, 1), (-1, -1), 8),
    ("TOPPADDING", (0, 1), (-1, -1), 8),
    ("GRID", (0, 0), (-1, -1), 0.75, colors.black),
]))

story.append(table)

# -----------------------------------------------------------------------------
# 6. Add rough matching rule immediately after table
# -----------------------------------------------------------------------------
story.append(Spacer(1, 0.25 * inch))

story.append(Paragraph("Rough Matching Rule:", note_heading_style))

story.append(Paragraph(
    "BRCA and BLCA are considered roughly matching if the absolute difference "
    "between My Accuracy and Paper Accuracy is less than or equal to 0.03.",
    note_style
))

story.append(Paragraph(
    "OV is considered roughly matching if the absolute difference between "
    "My Accuracy and Paper Accuracy is less than or equal to 0.06, because "
    "OV shows higher variation across runs.",
    note_style
))

# Build PDF
doc.build(story)

print("\nPDF created successfully:")
print(pdf_path)

file_size_kb = os.path.getsize(pdf_path) / 1024
print(f"PDF size: {file_size_kb:.2f} KB")

# -----------------------------------------------------------------------------
# 7. Push PDF to GitHub
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("PUSHING PDF TO GITHUB")
print("=" * 80)

github_username = "goutamsaums"
github_token = getpass("Enter GitHub Personal Access Token: ")

repo_url_with_token = MY_REPO_URL.replace(
    "https://",
    f"https://{github_username}:{github_token}@"
)

os.chdir(MY_REPO_DIR)

try:
    subprocess.run(
        ["git", "remote", "set-url", "origin", repo_url_with_token],
        check=True
    )

    subprocess.run(["git", "config", "user.name", "GOUTAM ANAND"], check=True)
    subprocess.run(["git", "config", "user.email", "goutam@students.sau.ac.in"], check=True)

    pdf_relative_path = os.path.join(RESULT_FOLDER, "MAGNET_Reproduction_Accuracy_Report.pdf")
    subprocess.run(["git", "add", pdf_relative_path], check=True)

    status = subprocess.run(
        ["git", "status", "--porcelain"],
        capture_output=True,
        text=True
    )

    if status.stdout.strip():
        subprocess.run(
            ["git", "commit", "-m", "Add MAGNET reproduction accuracy PDF report"],
            check=True
        )
        subprocess.run(["git", "push"], check=True)
        print("\nPDF report pushed to GitHub successfully.")
    else:
        print("\nNo changes to commit. PDF may already be up to date.")

finally:
    subprocess.run(
        ["git", "remote", "set-url", "origin", MY_REPO_URL],
        check=False
    )

# -----------------------------------------------------------------------------
# 8. Final confirmation
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CELL 11 COMPLETE")
print("=" * 80)
print("PDF saved at:")
print(pdf_path)

print("\nGitHub folder:")
print("https://github.com/goutamsaums/mlsi-summer-internship-2026/tree/main/MAGNET_Reproduction_BRCA_BLCA_OV")

print("\nPDF file name:")
print("MAGNET_Reproduction_Accuracy_Report.pdf")
print("=" * 80)

CELL 11: GENERATE PDF COMPARISON REPORT AND PUSH TO GITHUB

Final table to be written in PDF:


,Dataset,My Accuracy,Paper Accuracy,Roughly Matches?
0,BRCA,0.9198,0.918,Yes
1,BLCA,0.9705,0.970,Yes
2,OV,0.5753,0.614,Yes



Installing ReportLab if needed...

PDF created successfully:
/content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/MAGNET_Reproduction_Accuracy_Report.pdf
PDF size: 2.17 KB

PUSHING PDF TO GITHUB
Enter GitHub Personal Access Token: ··········

PDF report pushed to GitHub successfully.

CELL 11 COMPLETE
PDF saved at:
/content/mlsi-summer-internship-2026/MAGNET_Reproduction_BRCA_BLCA_OV/MAGNET_Reproduction_Accuracy_Report.pdf

GitHub folder:
https://github.com/goutamsaums/mlsi-summer-internship-2026/tree/main/MAGNET_Reproduction_BRCA_BLCA_OV

PDF file name:
MAGNET_Reproduction_Accuracy_Report.pdf
